In [ ]:
!pip install pandas --no-deps --ignore-installed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 27.3 MB/s  0:00:00


In [ ]:
!pip install apify-client -q

In [ ]:
from apify_client import ApifyClient
import pandas as pd
import time

client = ApifyClient(APIFY_TOKEN)
CITIES = {
    "tuapse": "Туапсе",
    "yeisk": "Ейск",
}

EXCLUDE_KEYWORDS = ['apart', 'апартамент', 'хостел', 'hostel', 'пансионат', 'квартир', 'flat', 'loft', 'гостевой дом']
all_reviews = []

for eng_city, rus_city in CITIES.items():
    hotels_run = client.actor("getascraper/ostrovok-hotels-scraper").call(run_input={
        "city": eng_city, "maxItems": 20, "language": "ru"
    })
    hotels = list(client.dataset(hotels_run.default_dataset_id).iterate_items())
    hotel_dict = {}
    for h in hotels:
        name = h.get("name") or ""
        if not any(kw in name.lower() for kw in EXCLUDE_KEYWORDS):
            hotel_id = h.get("hotelId")
            if hotel_id:
                hotel_dict[hotel_id] = name

    hotel_ids = list(hotel_dict.keys())
    if not hotel_ids:
        continue
    reviews_run = client.actor("getascraper/ostrovok-reviews-scraper").call(run_input={
        "hotels": hotel_ids, "language": "ru", "withTextOnly": True, "maxItemsPerHotel": 100
    })
    reviews = list(client.dataset(reviews_run.default_dataset_id).iterate_items())
    for r in reviews:
        r["Регион"] = "Краснодарский край"
        r["Город"] = rus_city
        h_id = r.get("hotelId") or r.get("hotelSlug")
        r["Название_Отеля"] = hotel_dict.get(h_id, h_id if h_id else "Не указан")
        all_reviews.append(r)
    time.sleep(2)

if all_reviews:
    df = pd.DataFrame(all_reviews)
    df['reviewDate'] = pd.to_datetime(df['reviewDate'], errors='coerce')
    df = df[(df['reviewDate'].dt.year >= 2024) & (df['reviewDate'].dt.year <= 2026)].copy()
    df['Дата'] = df['reviewDate'].dt.strftime('%d.%m.%Y')
    df['Отель'] = df['Название_Отеля']
    rename_map = {'rating': 'Рейтинг', 'author': 'Автор', 'text': 'Отзыв'}
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    if 'Рейтинг' not in df.columns:
        df['Рейтинг'] = None

    final_cols = ['Регион', 'Город', 'Отель', 'Рейтинг', 'Автор', 'Отзыв', 'Дата']
    df_final = df[final_cols].copy()

    df_final.to_csv("krasnodar2.csv", index=False, encoding="utf-8-sig")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
[apify.ostrovok-hotels-scraper runId:flP5Z0j79c0nJ8kqf] -> Status: RUNNING, Message: 
[apify.ostrovok-hotels-scraper runId:flP5Z0j79c0nJ8kqf] -> 2026-11-08T13:24:47.295Z ACTOR: Pulling container image of build 66J1cZrJN48de7pN3 from registry.
[apify.ostrovok-hotels-scraper runId:flP5Z0j79c0nJ8kqf] -> 2026-11-08T13:24:47.297Z ACTOR: Creating container.
[apify.ostrovok-hotels-scraper runId:flP5Z0j79c0nJ8kqf] -> 2026-11-08T13:24:47.355Z ACTOR: Starting container.
[apify.ostrovok-hotels-scraper runId:flP5Z0j79c0nJ8kqf] -> 2026-11-08T13:24:47.357Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.ostrovok-hotels-scraper runId:flP5Z0j79c0nJ8kqf] -> 2026-11-08T13:24:48.243Z INFO  System info {"apifyVersion":"3.7.2","apifyClientVersion":"2.25.0","crawleeVersion":"3.18.2","osType":"Linux

In [ ]:
import pandas as pd
df = pd.read_csv("krasnodar5.csv")
final_cols = ['Регион', 'Город', 'Отель', 'Рейтинг', 'Автор', 'Отзыв', 'Дата']
available_cols = [c for c in final_cols if c in df.columns]
df_d = df[available_cols].copy()
df_d.to_csv("krasnodar3.csv", index=False, encoding="utf-8-sig")
df_d.head()

,Регион,Город,Отель,Рейтинг,Автор,Отзыв,Дата
0,Краснодарский край,Геленджик,Отель Гранд Премьер,9.7,Vadim,Отдыхали родители четыре дня с питанием. Очень...,2026-09-24
1,Краснодарский край,Геленджик,Отель Гранд Премьер,9.7,Natalya,"Отель классный , номера чистые уютные , балкон...",2026-07-31
2,Краснодарский край,Геленджик,Отель Гранд Премьер,10.0,Anna,Захотелось на сутки сменить обстановку. Выбор ...,2026-03-21
3,Краснодарский край,Геленджик,Отель Гранд Премьер,9.9,Alina,"Отдыхаем сейчас в Гранд Премьере,номер комфорт...",2026-02-28
4,Краснодарский край,Геленджик,Отель Гранд Премьер,10.0,Svetlana,"Все понравилось! 😻 стандарт, но в номере все е...",2024-12-07


In [ ]:
df_d.shape

(1852, 7)

In [ ]:
df_d.tail()

,Регион,Город,Отель,Рейтинг,Автор,Отзыв,Дата
1847,Краснодарский край,Ейск,Отель Бристоль,7.9,Evgenii,"Хорошее расположение, в самом центре. Мы были ...",29.03.2025
1848,Краснодарский край,Ейск,Отель Бристоль,7.0,Vera,Расположен отель в центре города. До нормально...,06.08.2026
1849,Краснодарский край,Ейск,Отель Бристоль,6.6,Dmitrii,Расположение и парковка\n\nСтарые убитые номер...,27.06.2026
1850,Краснодарский край,Ейск,Отель Бристоль,6.8,Ekaterina,"удобное расположение в центре ,входящий шведск...",15.06.2026
1851,Краснодарский край,Ейск,Дом Двухэтажный в 5 Минутах от Моря,9.0,Elena,"Отдыхали с семьей , месторасположение отличное...",21.06.2025


здесь очистка от не6нужных объектов(все что не относится к отелям)

In [ ]:
df = pd.read_csv("krasnodar3.csv")
words = [
    'дом', 'квартира', 'апартамент', 'хостел', 'гостевой',
    'коттедж', 'студия', 'комната', 'жилой', 'flat', 'apart',
    'hostel', 'loft', 'гостевой дом'
]
mask = ~df['Отель'].astype(str).str.lower().str.contains('|'.join(bad_words), na=False)
df_h = df[mask].copy()
df_h = df_h.drop_duplicates(subset=['Отзыв'])
df_h.to_csv("krasnodar4.csv", index=False, encoding="utf-8-sig")
df_h['Отель'].value_counts().head(10)

Отель
Отель Гранд Премьер     100
Отель Атлас             100
Отель Oscar             100
Отель Vogue             100
Старинная Анапа         100
Отель Премьер            97
Отель Евразия            88
Отель ГК "Альбатрос"     74
Отель Абрикос Плюс       68
Отель Астель Вилла       67
Name: count, dtype: int64

In [ ]:
df_h.shape

(1844, 7)

In [ ]:
df_h.tail()

,Регион,Город,Отель,Рейтинг,Автор,Отзыв,Дата
1846,Краснодарский край,Ейск,Отель Бристоль,8.6,Aleksandr,В номере полулюкс телевизор Polar с диагональю...,05.05.2026
1847,Краснодарский край,Ейск,Отель Бристоль,7.9,Evgenii,"Хорошее расположение, в самом центре. Мы были ...",29.03.2025
1848,Краснодарский край,Ейск,Отель Бристоль,7.0,Vera,Расположен отель в центре города. До нормально...,06.08.2026
1849,Краснодарский край,Ейск,Отель Бристоль,6.6,Dmitrii,Расположение и парковка\n\nСтарые убитые номер...,27.06.2026
1850,Краснодарский край,Ейск,Отель Бристоль,6.8,Ekaterina,"удобное расположение в центре ,входящий шведск...",15.06.2026


In [ ]:
df1 = pd.read_csv("krasnodar.csv")
df2 = pd.read_csv("krasnodar4.csv")
def clean_columns(df):
    return df.rename(columns={
        'Название_Отеля': 'Отель', 'hotel': 'Отель', 'hotelName': 'Отель', 'name': 'Отель',
        'rating': 'Рейтинг', 'author': 'Автор',
        'text': 'Отзыв', 'review_text': 'Отзыв',
        'reviewDate': 'Дата', 'date': 'Дата'
    })

df1 = clean_columns(df1)
df2 = clean_columns(df2)
df_all = pd.concat([df1, df2], ignore_index=True)
df_all = df_all.drop_duplicates(subset=['Отзыв'])
words = [
    'дом', 'квартира', 'апартамент', 'хостел', 'гостевой',
    'коттедж', 'студия', 'комната', 'жилой', 'flat', 'apart',
    'hostel', 'loft', 'гостевой дом'
]
mask = ~df_all['Отель'].astype(str).str.lower().str.contains('|'.join(bad_words), na=False)
df_all = df_all[mask].copy()
fi_cols = ['Регион', 'Город', 'Отель', 'Рейтинг', 'Автор', 'Отзыв', 'Дата']
available_cols = [c for c in fi_cols if c in df_all.columns]
df_fi = df_all[available_cols].copy()
df_fi.to_csv("krasnodar5.csv", index=False, encoding="utf-8-sig")
display(df_fi.shape)
display(df_fi['Город'].value_counts())
display(df_fi['Отель'].head(10))

(3101, 7)

Город
Краснодар       1151
Геленджик        961
Анапа            598
Туапсе           228
Новороссийск     106
Ейск              57
Name: count, dtype: int64

0    Отель City Park Hotel Sochi
1    Отель City Park Hotel Sochi
2    Отель City Park Hotel Sochi
3    Отель City Park Hotel Sochi
4    Отель City Park Hotel Sochi
5    Отель City Park Hotel Sochi
6    Отель City Park Hotel Sochi
7    Отель City Park Hotel Sochi
8    Отель City Park Hotel Sochi
9    Отель City Park Hotel Sochi
Name: Отель, dtype: str

In [ ]:

client = ApifyClient(APIFY_TOKEN)
CITIES = {
    "sochi": "Сочи"
}

EXCLUDE_KEYWORDS = ['apart', 'апартамент', 'хостел', 'hostel', 'дом','пансионат', 'квартир', 'flat', 'loft', 'гостевой дом']
all_reviews = []

for eng_city, rus_city in CITIES.items():
    hotels_run = client.actor("getascraper/ostrovok-hotels-scraper").call(run_input={
        "city": eng_city, "maxItems": 10, "language": "ru"
    })
    hotels = list(client.dataset(hotels_run.default_dataset_id).iterate_items())
    hotel_dict = {}
    for h in hotels:
        name = h.get("name") or ""
        if not any(kw in name.lower() for kw in EXCLUDE_KEYWORDS):
            hotel_id = h.get("hotelId")
            if hotel_id:
                hotel_dict[hotel_id] = name

    hotel_ids = list(hotel_dict.keys())
    if not hotel_ids:
        continue
    reviews_run = client.actor("getascraper/ostrovok-reviews-scraper").call(run_input={
        "hotels": hotel_ids, "language": "ru", "withTextOnly": True, "maxItemsPerHotel": 100
    })
    reviews = list(client.dataset(reviews_run.default_dataset_id).iterate_items())
    for r in reviews:
        r["Регион"] = "Краснодарский край"
        r["Город"] = rus_city
        h_id = r.get("hotelId") or r.get("hotelSlug")
        r["Название_Отеля"] = hotel_dict.get(h_id, h_id if h_id else "Не указан")
        all_reviews.append(r)
    time.sleep(2)

if all_reviews:
    df = pd.DataFrame(all_reviews)
    df['reviewDate'] = pd.to_datetime(df['reviewDate'], errors='coerce')
    df = df[(df['reviewDate'].dt.year >= 2024) & (df['reviewDate'].dt.year <= 2026)].copy()
    df['Дата'] = df['reviewDate'].dt.strftime('%d.%m.%Y')
    df['Отель'] = df['Название_Отеля']
    rename_map = {'rating': 'Рейтинг', 'author': 'Автор', 'text': 'Отзыв'}
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    if 'Рейтинг' not in df.columns:
        df['Рейтинг'] = None

    final_cols = ['Регион', 'Город', 'Отель', 'Рейтинг', 'Автор', 'Отзыв', 'Дата']
    df_final = df[final_cols].copy()

    df_final.to_csv("krasnodar9.csv", index=False, encoding="utf-8-sig")

[apify.ostrovok-hotels-scraper runId:alGVoq81yaLjlX5kg] -> Status: RUNNING, Message: 
[apify.ostrovok-hotels-scraper runId:alGVoq81yaLjlX5kg] -> 2026-11-08T14:05:19.938Z ACTOR: Pulling container image of build 66J1cZrJN48de7pN3 from registry.
[apify.ostrovok-hotels-scraper runId:alGVoq81yaLjlX5kg] -> 2026-11-08T14:05:19.940Z ACTOR: Creating container.
[apify.ostrovok-hotels-scraper runId:alGVoq81yaLjlX5kg] -> 2026-11-08T14:05:20.029Z ACTOR: Starting container.
[apify.ostrovok-hotels-scraper runId:alGVoq81yaLjlX5kg] -> 2026-11-08T14:05:20.030Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.ostrovok-hotels-scraper runId:alGVoq81yaLjlX5kg] -> 2026-11-08T14:05:21.155Z INFO  System info {"apifyVersion":"3.7.2","apifyClientVersion":"2.25.0","crawleeVersion":"3.18.2","osType":"Linux","nodeVersion":"v24.22.1"}
[apify.ostrovok-hotels-scraper runId:alGVoq81yaLjlX5kg] -> 2026-11-08T14:05:21.509Z INFO  Resolving region for "sochi"...
[apify.ostrovok-hotels-scraper runId:alGVoq81yaLjlX5kg] -> 2

In [ ]:
df2 = pd.read_csv("krasnodar9.csv")
df.head()

,reviewId,hotelSlug,hotelTotalReviews,Автор,authorResidency,reviewDate,Рейтинг,textPositive,textNegative,Отзыв,...,hotelReply,hasHotelReply,images,imagesCount,scrapedAt,Регион,Город,Название_Отеля,Дата,Отель
0,2258631,hotel_azimut_park_hotel_plyus_rezidens_sochi,53,Marina,en-ru,2026-07-28 10:02:38.700781+00:00,10.0,"Хороший отель, красивый! Цель поездки была про...",Немного бы обучить персонал быть повежливее к ...,"Хороший отель, красивый! Цель поездки была про...",...,"Добрый день, Марина, спасибо за подробные комм...",True,[https://cdn.ostrovok.ru/t/1024x768/hotel_revi...,4.0,2026-11-08T14:05:43.506Z,Краснодарский край,Сочи,AZIMUT Парк Отель Виноградная Сочи,28.07.2026,AZIMUT Парк Отель Виноградная Сочи
1,2253940,hotel_azimut_park_hotel_plyus_rezidens_sochi,53,Ekaterina,en-ru,2026-07-26 09:59:39.858922+00:00,9.7,"Отличный отель, очень чистенько, прекрасное и ...",Не нашли минусов,"Отличный отель, очень чистенько, прекрасное и ...",...,"Уважаемая Екатерина! Спасибо, что посетили наш...",True,[https://cdn.ostrovok.ru/t/1024x768/hotel_revi...,4.0,2026-11-08T14:05:43.507Z,Краснодарский край,Сочи,AZIMUT Парк Отель Виноградная Сочи,26.07.2026,AZIMUT Парк Отель Виноградная Сочи
2,2358541,hotel_azimut_park_hotel_plyus_rezidens_sochi,53,Aleksey,en-ru,2026-09-22 12:26:41.299746+00:00,9.7,"В целом всё отлично, за исключением мелких нед...",NaN,"В целом всё отлично, за исключением мелких нед...",...,"Здравствуйте, Алексей! Благодарим за отзыв и в...",True,NaN,NaN,2026-11-08T14:05:43.507Z,Краснодарский край,Сочи,AZIMUT Парк Отель Виноградная Сочи,22.09.2026,AZIMUT Парк Отель Виноградная Сочи
3,2313309,hotel_azimut_park_hotel_plyus_rezidens_sochi,53,Aleksandra,en-ru,2026-08-24 07:57:52.822135+00:00,10.0,Отдыхали с парнем в этом отеле с 12 по 21 авгу...,NaN,Отдыхали с парнем в этом отеле с 12 по 21 авгу...,...,"Здравствуйте, Александра! Благодарим вас за те...",True,NaN,NaN,2026-11-08T14:05:43.507Z,Краснодарский край,Сочи,AZIMUT Парк Отель Виноградная Сочи,24.08.2026,AZIMUT Парк Отель Виноградная Сочи
4,2029412,hotel_azimut_park_hotel_plyus_rezidens_sochi,53,Anastasiya,en-ru,2026-03-03 22:04:59.508643+00:00,9.3,"После ремонта - отель просто восторг! Чисто, с...",Горничные утром 2 раза «врывались» без стука в...,"После ремонта - отель просто восторг! Чисто, с...",...,"Уважаемая Anastasiya, \nблагодарим Вас за высо...",True,NaN,NaN,2026-11-08T14:05:43.507Z,Краснодарский край,Сочи,AZIMUT Парк Отель Виноградная Сочи,03.03.2026,AZIMUT Парк Отель Виноградная Сочи


In [ ]:
df.shape

(431, 33)

In [ ]:
final_cols = ['Регион', 'Город', 'Отель', 'Рейтинг', 'Автор', 'Отзыв', 'Дата']
available_cols = [c for c in final_cols if c in df.columns]

df_sochi = df[available_cols].copy()

df_sochi.to_csv("sochi.csv", index=False, encoding="utf-8-sig")

In [ ]:
df_sochi.shape

(431, 7)

In [ ]:
df1 = pd.read_csv("krasnodar5.csv")
df2 = pd.read_csv("krasnodar9.csv")
df3 = pd.read_csv("sochi.csv")
rename_map = {
    'Название_Отеля': 'Отель', 'hotel': 'Отель', 'hotelName': 'Отель', 'name': 'Отель',
    'rating': 'Рейтинг', 'author': 'Автор',
    'text': 'Отзыв', 'review_text': 'Отзыв',
    'reviewDate': 'Дата', 'date': 'Дата'
}

df1 = df1.rename(columns={k: v for k, v in rename_map.items() if k in df1.columns})
df2 = df2.rename(columns={k: v for k, v in rename_map.items() if k in df2.columns})
df3 = df3.rename(columns={k: v for k, v in rename_map.items() if k in df3.columns})
df4 = pd.concat([df1, df2, df3], ignore_index=True)

df4 = df4.drop_duplicates(subset=['Отзыв'])
bad_words = [
    'дом', 'квартира', 'апартамент', 'хостел', 'гостевой',
    'коттедж', 'студия', 'комната', 'жилой', 'flat', 'apart',
    'hostel', 'loft'
]
mask = ~df4['Отель'].astype(str).str.lower().str.contains('|'.join(bad_words), na=False)
df4 = df4[mask].copy()
df4.loc[df4['Отель'].str.contains('Сочи|Sochi', case=False, na=False), 'Город'] = 'Сочи'
final_cols = ['Регион', 'Город', 'Отель', 'Рейтинг', 'Автор', 'Отзыв', 'Дата']
available_cols = [c for c in final_cols if c in df4.columns]
df_final = df4[available_cols].copy()

df_final.to_csv("krasnodar3.csv", index=False, encoding="utf-8-sig")
df_final['Город'].value_counts()

Город
Геленджик       961
Краснодар       763
Анапа           598
Сочи            420
Туапсе          228
Новороссийск    106
Ейск             57
Name: count, dtype: int64

In [ ]:
df_final.shape

(3133, 7)

In [ ]:
df_final.head()

,Регион,Город,Отель,Рейтинг,Автор,Отзыв,Дата
0,Краснодарский край,Сочи,Отель City Park Hotel Sochi,10.0,Darya,"Отличный отель, близко к морю, к морпорту, гул...",04.07.2026
1,Краснодарский край,Сочи,Отель City Park Hotel Sochi,9.0,Olga,"Прекрасный персонал, вежливые, отзывчивые\n\nВ...",05.10.2025
2,Краснодарский край,Сочи,Отель City Park Hotel Sochi,9.8,Alexey,"Отель очень удачно расположен торцом к морю, з...",09.05.2025
3,Краснодарский край,Сочи,Отель City Park Hotel Sochi,9.1,Artyom,Местоположение отличное. \r\nСпа комплекс: бан...,27.10.2026
4,Краснодарский край,Сочи,Отель City Park Hotel Sochi,10.0,Olga,"Все понравилось , бронировали номер одного уро...",02.09.2026
